<a href="https://colab.research.google.com/github/flexfengfeng/SMU-Hackathon-Prep/blob/main/track4_sustainability_agrilink.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌱 Track 4 · Sustainability
## AgriLink: Helping Smallholder Farmers Meet a Singapore Buyer's Standards, Every Week

**Coaching session · ~60 min**

### The sponsor's challenge
The **Ministry of Energy, Trade and Industry (METI)** asks:

> *"How might we use technology to help smallholder farmers improve produce quality and consistency, and supply reliability, so that they can meet larger buyers' requirements and raise their incomes?"*

### The real-world problem
In August 2026, APEDA and FairPrice Group ran an Indian Food Roadshow in Singapore: 160+ products from 18 Indian exporters. Shoppers liked them. But turning a roadshow into **regular shelf space** is hard:
- Some products **varied in quality** from batch to batch.
- Exporters were **not familiar with Singapore's import requirements**.
- A retailer needs the **same quality and the same volume every week**, and unpredictable weather makes that hard for small farms.

Behind each exporter sit hundreds of smallholders, organised into a **farmer producer organisation (FPO)**. Each farmer has 1 to 2 acres, a basic phone, patchy signal, and little warning when a storm is coming.

### The sustainability angle
Climate shocks (unseasonal rain, heatwaves) hit **quantity and quality** at the same time. And in India a large share of fresh vegetables is lost between farm and shop, mostly from heat and slow handling. Food that is grown but never eaten wastes water, fertiliser, fuel and the farmer's income. Better forecasting, grading and cold chain mean **less waste and more stable incomes**.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Standards | **Tree index (vectorless RAG)** + reasoning-based retrieval | Ask *"What do we need to export okra to Singapore?"* and get a checklist with cited sections |
| Check | **LLM-as-judge with evidence** | Every checklist item must be backed by a quote from the documents |
| Grade | **Multimodal + structured output** | A photo of a crate of okra ➜ counts, sizes and defects as JSON |
| Decide | **LLM perceives, Python decides** | Grade A / B / Reject decided by code, using thresholds extracted from the buyer's spec and validated |
| Supply | **Weather API + pandas** | Forecast rain and heat ➜ expected harvest ➜ will we fill the buyer's 2-tonne order? |
| Narrate | **Batch generation** | A Hindi + English WhatsApp advisory for farmers and a confidence note for the buyer, in one call |

> ⚠️ All documents, companies, villages and farmers in this notebook are **fictional and illustrative**. The standards documents are written *in the style of* SFA and APEDA guidance, but they are **not real legal or regulatory text**. Always check the official sources.

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

`pandas`, `matplotlib`, `Pillow` (images) and `requests` (weather API) are already in Colab. No vector database: this track uses a tree index instead.

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also set the client to wait and retry when the free tier says *"429: too many requests"*. Free-tier keys only allow about 5 requests per minute, so every team will hit this during the hackathon. If a cell seems to pause for up to a minute, it is waiting out the limit, not stuck.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Free-tier keys only allow a few requests per minute. These options make the SDK
# wait and retry any single request that gets "429: too many requests".
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=15, max_delay=60, http_status_codes=[429, 500, 503])

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(retry_options=RETRY))
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-3.5-flash-lite"

def ask_gemini(contents, config=None):
    """Call Gemini. Rate-limit retries are handled by the client (see RETRY above)."""
    return client.models.generate_content(model=MODEL, contents=contents, config=config)

## Step 3: The Standards Library (Mock Documents)
To sell okra (*bhindi*, lady's finger) to a Singapore supermarket, an exporter has to satisfy three sets of rules at once:

| Doc | Written by | Says |
|---|---|---|
| **A** | The buyer, fictional retailer *Merlion Fresh Mart* | What "good okra" means to them: grades, packing, temperature, weekly volume |
| **B** | Singapore's food regulator (style of SFA) | What any importer must do: licence, permit, residue limits, labels |
| **C** | India's export side (style of APEDA / AGMARK) | How Indian export okra is graded, and farm registration and residue monitoring |

Nobody on a 2-acre farm reads three documents like this. That is the gap AgriLink fills.

> ⚠️ These are **shortened, fictional, illustrative** documents. Numbers like residue limits are placeholders, not the legal values.

In [ ]:
LIBRARY = """
# A. Merlion Fresh Mart: Produce Specification for Fresh Okra (MFM-SPEC-OKR-03, ILLUSTRATIVE)

## A1. Scope and Product Description
This specification covers fresh okra (Abelmoschus esculentus), sold as loose and prepacked retail product in Merlion Fresh Mart stores.
Supplier means the exporter holding the supply agreement. The supplier is responsible for every farm and pack-house in its supply base.

## A2. Quality Grades
Grades are assessed at the pack-house on a sample from each lot (see A3). Length is measured from the base of the cap to the tip, excluding the stalk.

### A2.1 Grade A (Retail Premium)
Pods must be fresh, tender (the tip snaps cleanly), uniform bright green and free of soil.
Pod length 7 to 12 cm. Pod diameter not more than 1.8 cm.
Within one carton, the difference between the longest and shortest pod must not exceed 4 cm.
Defective pods (blemishes, bruising, insect damage, cuts) must not exceed 5% by count.
Over-mature pods (pale or yellowing, thick, fibrous) must not exceed 2% by count.

### A2.2 Grade B (Foodservice)
Accepted for foodservice and ready-meal customers at a lower price.
Pod length 6 to 15 cm. Minor colour variation allowed.
Defective pods must not exceed 10% by count. Over-mature pods must not exceed 5% by count.

### A2.3 Reject
A lot that fails Grade B is rejected. Any lot with live insects, rot, mould or foreign matter is rejected regardless of other results.

## A3. Sampling and Inspection at the Pack-house
Take one sample of at least 100 pods per lot, or a photo-graded tray of at least 12 pods per 50 kg for screening.
Photo screening may be used to decide which lots need a full manual inspection. Final grade is confirmed by a trained grader.
Record the grader's name, lot code, date, grade and the reason for any downgrade.

## A4. Packing and Labelling
Pack in clean, new, ventilated 2 kg cartons. Do not overfill; pods must not be crushed.
Each carton label must show: product name, grade, country of origin, supplier name and address, net weight, pack date and lot code.
Retail prepacks (250 g trays) must also carry the importer's name and address in Singapore.

## A5. Temperature and Cold Chain
Pre-cool pods to 8 to 10 C within 4 hours of harvest. Do not store below 7 C: okra suffers chilling injury (pitting, dark patches).
Keep the cold chain unbroken from pack-house to airport. Pulp temperature at receipt in Singapore must not exceed 12 C.
Lots received above this temperature may be downgraded or rejected.

## A6. Food Safety, Residues and Traceability
Each lot must have a pesticide residue test report from an accredited laboratory, dated within 14 days of shipment.
Residues must be within Singapore's maximum residue limits (MRLs). Only pesticides approved for okra in India may be used, following label rates and pre-harvest intervals.
The lot code must trace back to the farmer, village, harvest date and pack-house. Suppliers must be able to trace any lot within 4 hours.

## A7. Supply Commitment and Performance
Weekly order: 2000 kg of Grade A okra, delivered to the Merlion Fresh Mart distribution centre every Thursday.
Order fill rate must be at least 95% averaged over 4 weeks. On-time delivery must be at least 95%.
The supplier must notify any expected shortfall at least 72 hours before delivery. Unannounced shortfalls count double in the supplier scorecard.

## A8. Rejection, Claims and Payment
Rejected lots are reported within 24 hours of receipt with photos and the reason. The supplier pays freight on rejected lots.
Payment is made 30 days after accepted delivery. Suppliers with a scorecard above 90% for 6 months qualify for a 12-month supply agreement.

# B. Importing Fresh Fruits and Vegetables into Singapore: Summary (ILLUSTRATIVE, in the style of SFA guidance)

## B1. Who Can Import
Only a business registered in Singapore and holding a valid licence or registration from the food authority may import fresh fruits and vegetables for sale.
The overseas exporter cannot import directly; it must work with a licensed Singapore importer.

## B2. Import Permit for Each Consignment
The importer must apply for an import permit through the national trade declaration system before each consignment arrives.
The permit lists the product, quantity, country of origin and the exporter.

## B3. Documents that Travel with the Consignment
Keep the commercial invoice, packing list and air waybill for each consignment.
Where the authority asks for it, a health or phytosanitary certificate from the exporting country's competent authority must accompany the consignment.
Check the current requirements for your product before the first shipment, because they can change.

## B4. Pesticide Residues (MRLs)
Fresh produce must not contain pesticide residues above the maximum residue limits set in Singapore's food regulations.
Where no specific limit is set for a pesticide on a crop, a very low default limit applies, so unapproved pesticides are a high risk.
The importer is legally responsible for the safety of the food it imports, so most importers ask suppliers for residue test reports.

## B5. Labelling of Prepacked Produce
Prepacked food must be labelled in English with: the name of the food, net quantity, country of origin, and the name and address of the local importer, distributor or packer.
Loose produce sold unpacked does not need a full label, but the country of origin should be shown at the point of sale.

## B6. Inspection, Sampling and Hold-and-Test
Consignments may be inspected and sampled on arrival. Samples are tested for residues and contaminants.
Products from sources with poor compliance history may be held until test results are available (hold-and-test). Failed consignments may be rejected or destroyed at the importer's cost.

## B7. Record Keeping and Recall
Importers must keep import records and be able to trace each consignment to its source.
If a food safety problem is found, the importer must be able to recall the product quickly.

# C. Export Grading Guide for Fresh Okra: Summary (ILLUSTRATIVE, in the style of APEDA / AGMARK grading)

## C1. Grade Designations
Fresh okra for export is graded into three classes: Extra Class, Class I and Class II.
Extra Class: superior quality, uniform shape and colour, practically free of defects.
Class I: good quality, slight defects in shape and colour allowed, no damage to the flesh.
Class II: meets minimum requirements; defects allowed if the okra keeps its essential quality.

## C2. Minimum Requirements (all classes)
Pods must be whole, sound, clean, fresh in appearance, practically free of pests and pest damage, free of abnormal moisture and of any foreign smell or taste.
Pods must be harvested young and tender, before the seeds harden.

## C3. Size Coding
Size is based on pod length: small 6 to 9 cm, medium 9 to 12 cm, large 12 to 15 cm. Pods in one package must be of the same size code.

## C4. Tolerances
Extra Class: up to 5% by number not meeting the class requirements. Class I: up to 10%. Class II: up to 10% not meeting minimum requirements, but no rot.

## C5. Farm Registration and Residue Monitoring
Farms supplying export vegetables should be registered through the exporting state's horticulture department in the national traceability system.
Each registered farm keeps a spray diary (product, dose, date) and observes pre-harvest intervals. Samples are tested by an accredited laboratory under a residue monitoring plan.
An FPO can register its member farms as a group, and keep one shared spray record per farmer in a register or a phone app.

## C6. Pre-cooling and Pack-house Practice
Harvest in the early morning. Keep harvested pods in shade in ventilated crates, never in sacks.
Move pods to a pack-house within 4 hours. Pre-cool using forced air before grading and packing. Grade, pack and label at the pack-house, not in the field.
Small solar-powered cold rooms at village collection centres can cut losses where grid power is unreliable.
"""
print(f"Library length: {len(LIBRARY.split())} words")

## Step 4: Build a Tree Index (No Vectors!)
Standards documents are **already organised** by their authors: numbered sections and sub-sections. Chopping them into random chunks for a vector database would split a grade's rules from its heading, and lose which document a rule came from.

So, like an expert, we use the **table of contents**. Each heading becomes a node in a tree, holding its text and its child sections.

In [ ]:
import re

def build_tree(markdown):
    nodes, stack = [], []
    for line in markdown.strip().splitlines():
        m = re.match(r"^(#{1,4})\s+(.*)", line)
        if m:
            node = {"id": f"N{len(nodes):02d}", "level": len(m.group(1)), "title": m.group(2).strip(),
                    "text": "", "children": []}
            while stack and stack[-1]["level"] >= node["level"]:
                stack.pop()
            if stack:
                stack[-1]["children"].append(node["id"])
            stack.append(node)
            nodes.append(node)
        elif nodes:
            nodes[-1]["text"] += line + "\n"
    return {n["id"]: n for n in nodes}

def table_of_contents(tree):
    lines = []
    for n in tree.values():
        preview = " ".join(n["text"].split())[:70]
        lines.append(f'{"  " * (n["level"] - 1)}[{n["id"]}] {n["title"]}' + (f" | {preview}..." if preview else ""))
    return "\n".join(lines)

def node_text(tree, node_id, seen=None):
    """Return a section's text plus all of its sub-sections."""
    seen = set() if seen is None else seen
    if node_id in seen or node_id not in tree:
        return ""
    seen.add(node_id)
    n = tree[node_id]
    return f'### {n["title"]} [{node_id}]\n{n["text"]}' + "".join(node_text(tree, c, seen) for c in n["children"])

TREE = build_tree(LIBRARY)
TOC = table_of_contents(TREE)
print(f"{len(TREE)} sections across 3 documents\n")
print(TOC)

## Step 5: Ask the Standards: "What do we need to export okra to Singapore?"
Two LLM calls:
1. **Navigate:** the model reads only the table of contents and picks the sections to open, explaining why.
2. **Answer:** using only those sections, it writes a **compliance checklist** as structured output. Each item says *who* in the supply chain must do it (farmer, FPO, exporter or Singapore importer) and cites a section ID.

The *owner* field matters. A checklist that says "test for residues" is useless until someone knows it's the **farmer** who keeps the spray diary and the **exporter** who pays for the lab test.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Navigation(BaseModel):
    reasoning: str = Field(description="One or two sentences: why these sections are likely to hold the answer")
    node_ids: list[str] = Field(description="Up to 6 node IDs to open, e.g. ['N04']")

class ChecklistItem(BaseModel):
    requirement: str = Field(description="One short action, in plain words")
    category: Literal["Licence & permits", "Quality grade", "Food safety & residues",
                      "Packing & labelling", "Cold chain & logistics", "Supply commitment"]
    owner: Literal["Farmer", "FPO", "Exporter", "Singapore importer"]
    section_id: str = Field(description="The section ID that states this requirement, e.g. 'N05'")

def navigate(question, tree=None, max_sections=6):
    tree = tree or TREE
    prompt = f"""You are an agri-export compliance adviser. Here is the table of contents of three documents:
A = the Singapore buyer's specification, B = Singapore import rules, C = the Indian export grading guide.
Pick the sections (up to {max_sections}) you need to open to answer the question. Prefer specific sub-sections.

TABLE OF CONTENTS:
{table_of_contents(tree)}

QUESTION: {question}"""
    nav = ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json", response_schema=Navigation)).parsed
    nav.node_ids = [i for i in nav.node_ids if i in tree][:max_sections]  # never trust IDs blindly
    return nav

def ask_standards(question, tree=None):
    tree = tree or TREE
    nav = navigate(question, tree)
    seen = set()
    context = "".join(node_text(tree, i, seen) for i in nav.node_ids)
    prompt = f"""Answer the question as a checklist, using ONLY the sections below.
One item per distinct requirement (max 12). Cite the section ID where the requirement is stated.
Do not add requirements that are not in the sections.

SECTIONS:
{context}

QUESTION: {question}"""
    items = ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json",
                                                         response_schema=list[ChecklistItem])).parsed
    return nav, items

QUESTION = "What do we need to do to export fresh okra to Merlion Fresh Mart in Singapore?"
nav, checklist = ask_standards(QUESTION)
print("🧭 Sections opened:", [f"{i} {TREE[i]['title'][:40]}" for i in nav.node_ids])
print("   because:", nav.reasoning, "\n")

import pandas as pd
checklist_df = pd.DataFrame([c.model_dump() for c in checklist])
checklist_df["section"] = checklist_df.section_id.map(lambda i: TREE[i]["title"][:45] if i in TREE else "❓ unknown ID")
checklist_df.sort_values(["owner", "category"])

## Step 6: Check the Checklist (LLM-as-Judge with Evidence)
A wrong compliance checklist is worse than none: a shipment can be rejected at the border. So a second call acts as a **strict checker**. For each item it opens the cited section and must **quote the sentence** that supports it. No quote, no pass.

We also check in Python that every quote really appears in the documents. The judge can't make up evidence.

In [ ]:
class ItemCheck(BaseModel):
    requirement: str
    verdict: Literal["Supported", "Not supported"]
    evidence_quote: str = Field(description="A short exact quote from the cited section, or empty if none")

def judge_checklist(items, tree=None):
    tree = tree or TREE
    blocks = "\n".join(f"- ITEM: {it.requirement}\n  CITED SECTION [{it.section_id}]: "
                       f"{' '.join(tree.get(it.section_id, {'text': '(no such section)'})['text'].split())}"
                       for it in items)
    prompt = f"""You are a strict compliance checker. For each item, decide if its CITED SECTION supports it.
Supported: the section clearly states this requirement. Not supported: it does not, or says something different.
Copy a short exact quote (max 20 words) from the cited section as evidence.

{blocks}"""
    return ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json",
                                                        response_schema=list[ItemCheck])).parsed

flat_library = " ".join(LIBRARY.split()).lower()
checks = judge_checklist(checklist)
for c in checks:
    quote_ok = bool(c.evidence_quote) and " ".join(c.evidence_quote.split()).lower().strip(' ."') in flat_library
    icon = "🟢" if c.verdict == "Supported" and quote_ok else "🔴"
    print(f"{icon} {c.requirement}\n   📎 \"{c.evidence_quote}\"  {'(quote found ✅)' if quote_ok else '(quote NOT found in documents ❌)'}")

## Step 7: Extract the Grade Rules and Validate Them
To grade photos with **code**, we need the buyer's thresholds as numbers: Grade A length 7 to 12 cm, max 5% defects, and so on. We navigate to the grading sections and extract the numbers with structured output.

**Validation trick:** every number the model extracts must **literally appear** in the source sections. If the model "remembers" a number from somewhere else, or mixes up two grades, the check flags it. This is cheap, and it catches the most dangerous kind of error.

In [ ]:
class GradeRules(BaseModel):
    a_min_length_cm: float
    a_max_length_cm: float
    a_max_length_spread_cm: float = Field(description="Max difference between longest and shortest pod in a carton, Grade A")
    a_max_defect_pct: float
    a_max_overmature_pct: float
    b_min_length_cm: float
    b_max_length_cm: float
    b_max_defect_pct: float
    b_max_overmature_pct: float
    max_receipt_temp_c: float = Field(description="Max pulp temperature at receipt in Singapore")
    weekly_order_kg: float = Field(description="Weekly Grade A order in kg")

rules_nav = navigate("Grade A and Grade B thresholds (length, defects, over-mature), receipt temperature, and weekly order volume in the buyer's specification")
rules_context = "".join(node_text(TREE, i, set()) for i in rules_nav.node_ids)
print("🧭 Sections used:", rules_nav.node_ids)

RULES = ask_gemini(
    "Extract the buyer's grading thresholds from these sections. Copy numbers exactly as written.\n\n" + rules_context,
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=GradeRules),
).parsed

# Validation: every extracted number must appear in the sections we sent
numbers_in_source = {float(x) for x in re.findall(r"\d+(?:\.\d+)?", rules_context.replace(",", ""))}
rules_df = pd.DataFrame({"value": RULES.model_dump()})
rules_df["in_source"] = rules_df.value.map(lambda v: "✅ found" if v in numbers_in_source else "❌ not in source, check by hand")
rules_df

## Step 8: Grade a Photo of the Harvest (Multimodal + Structured Output)
At a village collection centre, a field agent puts a sample of pods on a grading tray and takes **one phone photo**. That's the whole workflow for the farmer.

We draw two sample trays with PIL: a **good lot** and a **mixed lot** (uneven sizes, a blemish, insect damage, a yellowing over-mature pod). Each tray has a **10 cm scale bar**. Without a reference object, no camera (or person) can judge size in cm.

Gemini **counts and measures** (perception). It also suggests a grade, but in the next step **Python decides**.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

PX_PER_CM = 20

def draw_pod(d, x, y, length_cm, colour=(70, 160, 40), spots=0, hole=False, thick=False):
    L, h = length_cm * PX_PER_CM, (34 if thick else 26)
    d.polygon([(x, y - h / 2), (x + L * 0.75, y - h * 0.4), (x + L, y), (x + L * 0.75, y + h * 0.4), (x, y + h / 2)],
              fill=colour, outline=(35, 80, 20))
    d.line([(x + 4, y), (x + L * 0.9, y)], fill=tuple(max(c - 40, 0) for c in colour), width=2)  # ridge
    d.rectangle([x - 16, y - h / 2 + 3, x, y + h / 2 - 3], fill=(50, 100, 30))                  # cap
    d.line([(x - 30, y), (x - 16, y)], fill=(90, 120, 50), width=4)                               # stalk
    for k in range(spots):
        d.ellipse([x + L * (0.3 + 0.15 * k) - 7, y - 8, x + L * (0.3 + 0.15 * k) + 7, y + 6], fill=(110, 70, 30))
    if hole:
        d.ellipse([x + L * 0.5 - 5, y - 5, x + L * 0.5 + 5, y + 5], fill=(30, 20, 10))

def make_tray(pods, title):
    img = Image.new("RGB", (1000, 680), (235, 232, 222))
    d = ImageDraw.Draw(img)
    for gx in range(0, 1000, 5 * PX_PER_CM):
        d.line([(gx, 40), (gx, 620)], fill=(215, 210, 198))
    d.text((20, 12), title, fill=(40, 40, 40))
    for k, pod in enumerate(pods):
        col, row = divmod(k, 7)
        draw_pod(d, 70 + col * 470, 85 + row * 75, **pod)
    d.line([(60, 650), (60 + 10 * PX_PER_CM, 650)], fill=(0, 0, 0), width=5)  # scale bar
    d.text((60 + 10 * PX_PER_CM + 10, 642), "10 cm", fill=(0, 0, 0))
    return img

rng = np.random.default_rng(4)
GOOD_TRAY = make_tray([{"length_cm": float(L)} for L in rng.uniform(8.5, 10.5, 14).round(1)],
                      "Lot NDG-1104-07 | grading tray")
mixed = [6.0, 13.5, 7.5, 11.0, 9.0, 14.0, 8.0, 12.5, 6.5, 10.0, 9.5, 13.0, 7.0, 11.5]
MIXED_TRAY = make_tray([{"length_cm": L} for L in mixed], "Lot KLP-1104-02 | grading tray")
draw = ImageDraw.Draw(MIXED_TRAY)  # add the problems on top: blemish, insect hole, over-mature pod
draw_pod(draw, 70, 85 + 3 * 75, 11.0, spots=3)
draw_pod(draw, 540, 85 + 0 * 75, 12.5, hole=True)
draw_pod(draw, 70, 85 + 5 * 75, 14.0, colour=(175, 170, 70), thick=True)

GOOD_TRAY.save("okra_good_lot.png"); MIXED_TRAY.save("okra_mixed_lot.png")
display(GOOD_TRAY.resize((500, 340))); display(MIXED_TRAY.resize((500, 340)))

In [ ]:
class PhotoGrade(BaseModel):
    pods_visible: int
    shortest_pod_cm: float = Field(description="Estimated length of the shortest pod, cap to tip, using the scale bar")
    longest_pod_cm: float
    pods_with_defects: int = Field(description="Pods with blemishes, bruises, holes, cuts or rot")
    pods_overmature: int = Field(description="Pods that are pale or yellowing, thick or fibrous-looking")
    colour: Literal["uniform bright green", "mixed green", "dull or yellowing"]
    defects_observed: list[str] = Field(description="Short descriptions of each problem seen")
    suggested_grade: Literal["A", "B", "Reject"]
    confidence: Literal["low", "medium", "high"]

def grade_photo(image, tree=None):
    tree = tree or TREE
    spec = "".join(node_text(tree, i, set()) for i, n in tree.items() if n["title"].startswith("A2."))
    prompt = f"""You are an okra quality grader at a village collection centre.
Look at the grading tray photo. Use the black 10 cm scale bar to estimate pod lengths (cap to tip, not the stalk).
Count every pod. Count defective and over-mature pods carefully. Then suggest a grade using the buyer's spec.
If the photo is blurry, has no scale bar, or is not okra, set confidence to low.

BUYER'S GRADE SPEC:
{spec}"""
    return ask_gemini([prompt, image], types.GenerateContentConfig(response_mime_type="application/json",
                                                                   response_schema=PhotoGrade)).parsed

photo_grades = {"Good lot": grade_photo(GOOD_TRAY), "Mixed lot": grade_photo(MIXED_TRAY)}  # 2 image calls
for name, g in photo_grades.items():
    print(f"📷 {name}:", g.model_dump_json(indent=2), "\n")

### 📸 Try it with your own photo (optional)
Put some vegetables on a table next to a ruler or a credit card (8.5 cm wide) and take a photo. This makes one extra LLM call.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick a .jpg or .png
if uploaded:
    photo = Image.open(next(iter(uploaded))).convert("RGB")
    photo.thumbnail((1024, 1024))  # smaller image = faster and cheaper
    display(photo)
    print(grade_photo(photo).model_dump_json(indent=2))

## Step 9: LLM Perceives, Python Decides
The model is good at **seeing** (counting pods, spotting a brown patch). It is less reliable at **applying thresholds** consistently: is 1 bad pod in 14 more or less than 5%?

So Python turns the counts into percentages and applies the **validated thresholds** from Step 7. Every downgrade comes with a reason that cites the spec clause. If Python and the model disagree, or the model's confidence is low, the lot goes to a **human grader**. That's how you build trust in quality assessment, which past agri-trading platforms found was essential.

In [ ]:
def rule_grade(g, r=None):
    """Apply the buyer's thresholds to the counts the model saw. Returns (grade, reasons)."""
    r = r or RULES
    n = max(g.pods_visible, 1)
    defect_pct, over_pct = 100 * g.pods_with_defects / n, 100 * g.pods_overmature / n
    spread = g.longest_pod_cm - g.shortest_pod_cm
    fails_a = [msg for bad, msg in [
        (g.shortest_pod_cm < r.a_min_length_cm or g.longest_pod_cm > r.a_max_length_cm,
         f"length {g.shortest_pod_cm:g}-{g.longest_pod_cm:g} cm outside {r.a_min_length_cm:g}-{r.a_max_length_cm:g} cm (A2.1)"),
        (spread > r.a_max_length_spread_cm, f"size spread {spread:.1f} cm > {r.a_max_length_spread_cm:g} cm (A2.1)"),
        (defect_pct > r.a_max_defect_pct, f"defects {defect_pct:.0f}% > {r.a_max_defect_pct:g}% (A2.1)"),
        (over_pct > r.a_max_overmature_pct, f"over-mature {over_pct:.0f}% > {r.a_max_overmature_pct:g}% (A2.1)"),
    ] if bad]
    fails_b = [msg for bad, msg in [
        (g.shortest_pod_cm < r.b_min_length_cm or g.longest_pod_cm > r.b_max_length_cm,
         f"length outside {r.b_min_length_cm:g}-{r.b_max_length_cm:g} cm (A2.2)"),
        (defect_pct > r.b_max_defect_pct, f"defects {defect_pct:.0f}% > {r.b_max_defect_pct:g}% (A2.2)"),
        (over_pct > r.b_max_overmature_pct, f"over-mature {over_pct:.0f}% > {r.b_max_overmature_pct:g}% (A2.2)"),
    ] if bad]
    if not fails_a:
        return "A", ["meets every Grade A threshold"]
    return ("B", fails_a) if not fails_b else ("Reject", fails_a + fails_b)

def grading_report(g):
    grade, reasons = rule_grade(g)
    needs_human = grade != g.suggested_grade or g.confidence == "low"
    return {"python_grade": grade, "model_suggested": g.suggested_grade, "confidence": g.confidence,
            "reasons": reasons, "action": "🧑‍🌾 send to human grader" if needs_human else "✅ auto-accept screening grade"}

for name, g in photo_grades.items():
    rep = grading_report(g)
    print(f"📷 {name}: Python says {rep['python_grade']}, model said {rep['model_suggested']} ({rep['confidence']}) ➜ {rep['action']}")
    for reason in rep["reasons"]:
        print("   -", reason)

### 🤔 Coaching discussion (5 min)
- The mixed lot fails Grade A. If the field agent **sorts out** the 3 bad pods and the extra-long ones at the collection centre, what grade could the rest reach? Quality improvement often happens at **sorting**, not on the farm.
- The photo is only a **screening** tool (spec A3 says a trained grader confirms). Who should that grader be: FPO staff, the exporter, or a third party both sides trust?
- What if the farmer's phone camera is poor, or the photo is taken at dusk? How would you design the workflow so a bad photo leads to "ask a human", never to a wrong grade?
- Which is worse for the farmer: an A lot wrongly called B, or a B lot wrongly called A that gets rejected in Singapore?

## Step 10: This Week's Weather for the FPO Villages
Weather is the biggest source of supply risk. We fetch a 7-day forecast from **[Open-Meteo](https://open-meteo.com/)** (free, no API key) for three fictional villages near real okra-growing districts.

`USE_LIVE_WEATHER = False` uses a **scripted storm week** so the demo is the same every time (unseasonal rain in Nashik, a heavy-rain day in Guntur, heat in Anand). Set it to `True` to see this week's real forecast. If the API is down or there is no internet, the notebook falls back to the scripted week.

In [ ]:
import requests

USE_LIVE_WEATHER = False  # True = real Open-Meteo forecast

VILLAGES = pd.DataFrame([  # fictional villages, coordinates of the real district towns
    {"village": "Nandgiri", "district": "Nashik, Maharashtra", "lat": 20.00, "lon": 73.79},
    {"village": "Kollapadu", "district": "Guntur, Andhra Pradesh", "lat": 16.31, "lon": 80.44},
    {"village": "Vasantpura", "district": "Anand, Gujarat", "lat": 22.56, "lon": 72.95},
])

def fetch_forecast(v):
    r = requests.get("https://api.open-meteo.com/v1/forecast", timeout=10, params={
        "latitude": v.lat, "longitude": v.lon, "daily": "precipitation_sum,temperature_2m_max",
        "forecast_days": 7, "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.DataFrame({"village": v.village, "date": d["time"], "rain_mm": d["precipitation_sum"], "tmax_c": d["temperature_2m_max"]})

def scripted_forecast():
    dates = pd.date_range(pd.Timestamp.today().normalize(), periods=7).strftime("%Y-%m-%d")
    week = {"Nandgiri":   ([0, 2, 28, 46, 9, 0, 0], [31, 30, 27, 26, 28, 30, 31]),
            "Kollapadu":  ([4, 0, 0, 1, 33, 12, 0], [32, 33, 33, 32, 29, 30, 32]),
            "Vasantpura": ([0, 0, 0, 0, 0, 0, 0], [34, 35, 37, 38, 36, 34, 33])}
    return pd.concat([pd.DataFrame({"village": v, "date": dates, "rain_mm": rain, "tmax_c": tmax})
                      for v, (rain, tmax) in week.items()], ignore_index=True)

WEATHER_SOURCE = "scripted storm week (synthetic)"
weather = scripted_forecast()
if USE_LIVE_WEATHER:
    try:
        weather = pd.concat([fetch_forecast(v) for v in VILLAGES.itertuples()], ignore_index=True)
        WEATHER_SOURCE = "Open-Meteo live forecast"
    except Exception as e:
        print("⚠️ Live forecast unavailable, using the scripted week instead:", e)
weather[["rain_mm", "tmax_c"]] = weather[["rain_mm", "tmax_c"]].fillna(0)

HEAVY_RAIN_MM, HOT_DAY_C = 20, 36  # okra: heavy rain = mud, rot, no field access; >36 C = fibrous, wilting pods
risk = weather.groupby("village").agg(rain_mm_week=("rain_mm", "sum"), max_temp_c=("tmax_c", "max"),
                                     heavy_rain_days=("rain_mm", lambda s: int((s >= HEAVY_RAIN_MM).sum())),
                                     hot_days=("tmax_c", lambda s: int((s >= HOT_DAY_C).sum())))
print("🌦️ Source:", WEATHER_SOURCE)
risk

## Step 11: Will We Fill the Order? (pandas Calculates)
The buyer wants **2 tonnes of Grade A every Thursday** (spec A7, the number we extracted and validated in Step 7). The exporter sources from **30 smallholders** in the three villages. For each farmer we know (synthetic data) this week's planned harvest and their **past Grade A share**.

The model is simple and every assumption is visible, so an FPO manager can argue with it:
1. **Weather risk** ➜ each heavy-rain day cuts yield 6% and Grade A share 8 points; each hot day cuts 4% and 5 points.
2. **Post-harvest loss** between field and Singapore: ~15% with no pre-cooling, ~6% with pre-cooling and an unbroken cold chain.
3. **Shortfall?** ➜ ask the most reliable farmers in the villages with the least rain for an **extra early picking**, before the storm.

> 📘 **Fill rate** = Grade A kg delivered ÷ kg ordered. The buyer's spec needs ≥ 95% over 4 weeks. All rates and prices below are **illustrative assumptions**, so replace them with real FPO data in a pilot.

In [ ]:
import matplotlib.pyplot as plt

ORDER_KG = RULES.weekly_order_kg
LOSS_NO_COLD, LOSS_PRECOOL = 0.15, 0.06              # share of harvest lost field ➜ shelf (illustrative)
PRICE_A, PRICE_B, PRICE_MANDI = 34, 24, 25             # ₹ per kg paid to farmer (illustrative)
PRECOOL_COST = 2.5                                     # ₹ per kg for pre-cooling + cold transport to airport

rng = np.random.default_rng(42)
farmers = pd.DataFrame({
    "farmer_id": [f"F{i:02d}" for i in range(1, 31)],
    "village": np.repeat(VILLAGES.village.values, 10),
    "planned_kg": rng.integers(70, 151, 30),            # this week's planned pickings
    "grade_a_hist": rng.uniform(0.6, 0.92, 30).round(2), # share of past lots graded A
})
f = farmers.merge(risk, left_on="village", right_index=True)
f["yield_factor"] = (1 - 0.06 * f.heavy_rain_days - 0.04 * f.hot_days).clip(0.6, 1)
f["expected_kg"] = (f.planned_kg * f.yield_factor).round()
f["grade_a_share"] = (f.grade_a_hist - 0.08 * f.heavy_rain_days - 0.05 * f.hot_days).clip(0.2, 0.95)
f["grade_a_kg"] = (f.expected_kg * f.grade_a_share).round()

plan_a = (f.planned_kg * f.grade_a_hist).sum()
after_weather = f.grade_a_kg.sum()
deliv_no_cold, deliv_precool = after_weather * (1 - LOSS_NO_COLD), after_weather * (1 - LOSS_PRECOOL)

# Pull forward: an extra early picking (~30% of a normal week) from reliable farmers in villages with < 2 heavy-rain days
shortfall = max(ORDER_KG - deliv_precool, 0)
cand = f[f.heavy_rain_days < 2].assign(extra_a_kg=lambda d: (0.3 * d.expected_kg * d.grade_a_share * (1 - LOSS_PRECOOL)).round())
cand = cand.sort_values("grade_a_share", ascending=False)
pull_forward = cand[cand.extra_a_kg.cumsum().shift(fill_value=0) < shortfall]
deliv_final = deliv_precool + pull_forward.extra_a_kg.sum()

scenarios = pd.Series({"Harvest plan\n(normal weather)": plan_a, "After weather\nrisk": after_weather,
                       "Delivered,\nno pre-cooling": deliv_no_cold, "Delivered,\nwith pre-cooling": deliv_precool,
                       "+ pull-forward\npickings": deliv_final}).round()
total_kg = f.expected_kg.sum()
SUPPLY = {
    "weather_source": WEATHER_SOURCE,
    "order_kg_grade_a": float(ORDER_KG),
    "fill_rate_no_precooling_pct": round(float(min(deliv_no_cold / ORDER_KG, 1)) * 100, 1),
    "fill_rate_with_precooling_pct": round(float(min(deliv_precool / ORDER_KG, 1)) * 100, 1),
    "fill_rate_with_pull_forward_pct": round(float(min(deliv_final / ORDER_KG, 1)) * 100, 1),
    "shortfall_before_pull_forward_kg": int(round(shortfall)),
    "farmers_to_pull_forward": pull_forward.farmer_id.tolist(),
    "post_harvest_loss_kg_no_precooling": int(round(total_kg * LOSS_NO_COLD)),
    "post_harvest_loss_kg_with_precooling": int(round(total_kg * LOSS_PRECOOL)),
    "precooling_cost_inr": int(round(total_kg * PRECOOL_COST)),
    "value_saved_by_precooling_inr": int(round((after_weather * PRICE_A + (total_kg - after_weather) * PRICE_B) * (LOSS_NO_COLD - LOSS_PRECOOL))),
}
# Farmer income this week: graded sale (minus pre-cooling cost) vs selling everything at the local mandi
a_kg, b_kg = deliv_precool, (total_kg - after_weather) * (1 - LOSS_PRECOOL)
SUPPLY["farmer_income_uplift_pct"] = round(float((a_kg * PRICE_A + b_kg * PRICE_B - total_kg * PRECOOL_COST) / (total_kg * (1 - LOSS_NO_COLD) * PRICE_MANDI) - 1) * 100, 1)

by_village = f.groupby("village").agg(farmers=("farmer_id", "count"), planned_kg=("planned_kg", "sum"),
                                      expected_kg=("expected_kg", "sum"), grade_a_kg=("grade_a_kg", "sum"),
                                      heavy_rain_days=("heavy_rain_days", "first"), hot_days=("hot_days", "first"))
by_village["grade_a_share_pct"] = (by_village.grade_a_kg / by_village.expected_kg * 100).round(1)
for k, v in SUPPLY.items():
    print(f"{k:38s} {v}")
by_village

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
for v, d in weather.groupby("village"):
    ax1.plot(pd.to_datetime(d.date).dt.strftime("%a %d"), d.rain_mm, "-o", label=v)
ax1.axhline(HEAVY_RAIN_MM, color="grey", ls=":", label="heavy rain")
ax1.set_title(f"Rain forecast (mm/day)\n{WEATHER_SOURCE}"); ax1.legend(fontsize=8); ax1.tick_params(axis="x", rotation=45)

colours = ["#9e9e9e", "#d95f02", "#e7298a", "#1b9e77", "#1b9e77"]
ax2.bar(scenarios.index, scenarios.values, color=colours)
ax2.axhline(ORDER_KG, color="black", ls="--", label=f"Order {ORDER_KG:,.0f} kg")
ax2.axhline(0.95 * ORDER_KG, color="black", ls=":", label="95% fill-rate target")
for i, val in enumerate(scenarios.values):
    ax2.text(i, val + 30, f"{val:,.0f}", ha="center", fontsize=9)
ax2.set_title("Grade A okra this week (kg)"); ax2.legend(fontsize=8, loc="lower left"); ax2.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

print("👩‍🌾 Pull-forward list:")
pull_forward[["farmer_id", "village", "grade_a_share", "extra_a_kg"]]

## Step 12: Turn the Numbers into Messages (LLM Narrates)
pandas did the maths. Now one LLM call writes for two very different readers:
- **Farmers**, per village: short, simple, voice-note friendly, in **Hindi and English**. Most smallholders read WhatsApp on a basic Android phone, and many prefer listening to reading. No percentages, no jargon: *what to do, and when*.
- **The Singapore buyer**: a calm, factual supply confidence note. Buyers commit to long contracts when they get **early, honest warnings** (spec A7 asks for 72 hours' notice of any shortfall).

We pass only the computed facts, so the model can't invent numbers.

In [ ]:
class VillageAdvisory(BaseModel):
    village: str
    english: str = Field(description="Max 50 words, very simple English, actions and days only")
    hindi: str = Field(description="The same message in simple Hindi (Devanagari script)")

class WeeklyMessages(BaseModel):
    advisories: list[VillageAdvisory]
    buyer_note: str = Field(description="Max 120 words for the Singapore buyer: expected fill rate, risks, actions, confidence")

def write_messages(by_village=by_village, supply=SUPPLY, pull_forward=pull_forward):
    prompt = f"""You write messages for AgriLink, which links an Indian okra exporter's smallholder farmers to a Singapore retailer.
Use ONLY the facts below. Do not invent numbers.

1) One WhatsApp advisory per village for farmers (low literacy, will be read aloud as a voice note):
   short sentences, no percentages, say what to do and which days. Mention: harvest early morning, keep pods in shade
   in crates (not sacks), bring to the collection centre within 4 hours. Villages with heavy rain: pick before the rain,
   don't pick wet pods. Hot villages: pick at dawn, water in the evening. Farmers on the pull-forward list are asked
   for one extra early picking (name them by farmer ID).
2) One note for the Singapore buyer.

VILLAGE FACTS:
{by_village.reset_index().to_dict(orient="records")}

SUPPLY FACTS:
{supply}

PULL-FORWARD FARMERS (farmer_id, village): {pull_forward[["farmer_id", "village"]].values.tolist()}"""
    return ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json", response_schema=WeeklyMessages)).parsed

MESSAGES = write_messages()
for a in MESSAGES.advisories:
    print(f"📱 {a.village}\n   EN: {a.english}\n   HI: {a.hindi}\n")
print("📨 Buyer note:\n", MESSAGES.buyer_note)

## Step 13: Build the Demo UI with Gradio 🏆
Three tabs, one per user:
- **Ask the standards** (exporter / FPO manager): tree-RAG checklist with cited sections. 2 LLM calls per question.
- **Grade a photo** (field agent): upload a tray photo; Gemini counts, Python grades. 1 LLM call.
- **This week's supply** (FPO + buyer): the chart, fill rate and messages, **precomputed** above, so 0 LLM calls.

In [ ]:
import gradio as gr

def ui_ask(question):
    if not question.strip():
        return [], ""
    try:
        nav, items = ask_standards(question)
    except Exception as e:
        return [], f"⚠️ Sorry, something went wrong: {e}"
    rows = [[it.owner, it.category, it.requirement, f"{it.section_id} {TREE.get(it.section_id, {'title': '?'})['title'][:40]}"] for it in items]
    path = "\n".join(f"- **{i}** {TREE[i]['title']}" for i in nav.node_ids)
    return rows, f"**Sections opened:**\n{path}\n\n**Why:** {nav.reasoning}"

def ui_grade(image):
    if image is None:
        return {}, "Please upload a photo of a grading tray."
    try:
        g = grade_photo(image)
    except Exception as e:
        return {}, f"⚠️ Sorry, something went wrong: {e}"
    rep = grading_report(g)
    reasons = "\n".join(f"- {r}" for r in rep["reasons"])
    return g.model_dump(), f"## Grade: {rep['python_grade']}\nModel suggested **{rep['model_suggested']}** ({rep['confidence']} confidence) ➜ {rep['action']}\n\n{reasons}"

supply_md = (f"### Fill rate: {SUPPLY['fill_rate_with_pull_forward_pct']}% of {ORDER_KG:,.0f} kg Grade A\n"
             f"(no pre-cooling: {SUPPLY['fill_rate_no_precooling_pct']}%, with pre-cooling: {SUPPLY['fill_rate_with_precooling_pct']}%)  \n"
             f"Post-harvest loss: {SUPPLY['post_harvest_loss_kg_no_precooling']:,} kg ➜ {SUPPLY['post_harvest_loss_kg_with_precooling']:,} kg with pre-cooling  \n"
             f"Pull forward: {', '.join(SUPPLY['farmers_to_pull_forward']) or 'none needed'}  \n"
             f"*Weather: {WEATHER_SOURCE}*\n\n### 📨 Note to buyer\n{MESSAGES.buyer_note}\n\n### 📱 Farmer advisories\n"
             + "\n".join(f"**{a.village}**  \nEN: {a.english}  \nHI: {a.hindi}\n" for a in MESSAGES.advisories))

with gr.Blocks(title="AgriLink") as demo:
    gr.Markdown("# 🌱 AgriLink\nQuality, standards and supply planning for an okra FPO selling to a Singapore retailer. *(All data fictional.)*")
    with gr.Tab("📚 Ask the standards"):
        q = gr.Textbox(label="Question", placeholder="e.g. What must be on the carton label?")
        gr.Examples(["What do we need to export okra to Singapore?", "What must be printed on each carton?",
                     "What temperature must the okra be when it arrives?", "What does the farmer need to record about pesticides?"], inputs=q)
        ask_btn = gr.Button("Ask", variant="primary")
        table = gr.Dataframe(headers=["Owner", "Category", "Requirement", "Section"], wrap=True)
        trail = gr.Markdown()
        ask_btn.click(ui_ask, inputs=q, outputs=[table, trail])
    with gr.Tab("📷 Grade a photo"):
        with gr.Row():
            with gr.Column():
                img_in = gr.Image(type="pil", label="Grading tray photo (include a scale bar or ruler)")
                gr.Examples(["okra_good_lot.png", "okra_mixed_lot.png"], inputs=img_in)
                grade_btn = gr.Button("Grade", variant="primary")
            with gr.Column():
                verdict = gr.Markdown()
                seen = gr.JSON(label="What the model saw")
        grade_btn.click(ui_grade, inputs=img_in, outputs=[seen, verdict])
    with gr.Tab("🚚 This week's supply"):
        gr.Plot(value=fig)
        gr.Markdown(supply_md)
        gr.Dataframe(value=by_village.reset_index())

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🎯 What the Sponsor Wants After 36 Hours
METI's strong outcome is a **pilot proposal**, not just an app. It must account for:
- [ ] **Production reliability**: how you forecast and protect weekly volumes (Steps 10 to 11).
- [ ] **Quality compliance**: how you grade consistently and meet Singapore's import and buyer rules (Steps 5 to 9).
- [ ] **Reduced post-harvest losses**: what changes physically (pre-cooling, crates, timing), and how you measure it.
- [ ] **Measurable indicators** with a baseline and a target.
- [ ] **Adoption at scale**: affordable, works within existing supply chains, sustainable operating model.

Copy this template into your pitch deck and fill it in:

**Pilot proposal: AgriLink** *(team name)*

| Item | Your answer |
|---|---|
| Crop and corridor | e.g. fresh okra, Nashik ➜ Singapore |
| Partners | FPO(s), exporter, Singapore importer / retailer, lab, cold-chain provider |
| Pilot size and length | e.g. 1 FPO, 300 farmers, 3 collection centres, 16 weeks |
| What farmers use | e.g. WhatsApp voice advisories + village agent with one phone |
| What the FPO / exporter uses | e.g. grading app, weekly supply plan, compliance checklist |
| Physical changes | e.g. plastic crates, shade at collection centre, solar cold room |
| Who pays, and how much | e.g. ₹0.50/kg buyer fee + FPO subscription |

| KPI | How measured | Baseline | Pilot target |
|---|---|---|---|
| % of lots meeting Grade A spec | Pack-house grading records | ? % | +15 points |
| Post-harvest loss % (field ➜ Singapore DC) | Weighbridge at collection centre vs DC receipt | ? % | halve it |
| Order fill rate (4-week average) | Buyer receipts vs orders | ? % | ≥ 95% |
| On-time delivery | Buyer receipts | ? % | ≥ 95% |
| Rejections at Singapore receipt | Buyer claims | ? lots | 0 |
| Farmer income uplift | Net price × saleable kg vs mandi baseline | ₹ ? | +20% |
| Cost per farmer per season | Total pilot cost ÷ farmers | ₹ ? | < ₹ ? |
| Photo grade agreement with human grader | Side-by-side checks | n/a | ≥ 90% |

**Scaling path:** 1 FPO, 1 crop, 1 buyer ➜ 3 FPOs and 2 crops (e.g. add mango or onion) ➜ other Singapore buyers and other export markets ➜ hand over to the FPO federation or a state agency once it pays for itself.

## ✅ Constraints Checklist
Each must-have from the brief, how this prototype meets it (✅), and the gap to close in your pitch (⚠️).
- [ ] **Affordable.** ✅ Free-tier LLM, free weather API, one phone per village, no new infrastructure apart from optional pre-cooling. ⚠️ Show real per-farmer cost, and that pre-cooling pays for itself (Step 11 compares its cost with the value saved).
- [ ] **Low connectivity.** ✅ Farmers only *receive* WhatsApp/SMS messages; the heavy work happens at the collection centre. ⚠️ Offline photo capture that syncs later; SMS fallback.
- [ ] **Low digital literacy.** ✅ Short, voice-note-friendly Hindi + English messages; a **village agent** takes photos and enters data for farmers. ⚠️ IVR calls in local languages (Marathi, Telugu, Gujarati); test the messages with real farmers.
- [ ] **Varying resources.** ✅ Works for farmers without a smartphone, because advice flows through the FPO and the agent. ⚠️ Include women farmers, tenant farmers and non-FPO members.
- [ ] **Works within existing supply chains.** ✅ Fits FPOs, collection centres, exporters and licensed Singapore importers; the local mandi stays the outlet for Grade B. ⚠️ Connect to India's farm registration / traceability system and to FPO records.
- [ ] **Whole supply chain** (quality assessment, processing, storage, transport). ✅ Grading, pre-cooling, transport loss and delivery are modelled. ⚠️ Packaging supply, air-freight space and customs timing.
- [ ] **Sustainable operating model.** ✅ The Grade A premium and the value saved by pre-cooling are calculated, so there is money to share. ⚠️ Decide who pays: a buyer fee per kg, an FPO subscription, an exporter licence, or bundling with input sales.
- [ ] **Trusted quality assessment.** ✅ Python applies the written spec, each downgrade cites a clause, and disagreements go to a human grader. ⚠️ Agree with the buyer that photo grades count for screening.
- [ ] **Not high-cost or specialist** (out of scope). ✅ No IoT sensors, drones or new cold-chain network required. ⚠️ Keep it that way as you add features.

> 💡 **Learn from history:** several Indian agri marketplaces and farm-advisory start-ups shut down when funding dried up, because they couldn't make money per transaction. Digital agri-trading pilots also showed that buyers only trust a platform when **quality assessment is trusted** and **logistics actually work**. Your pitch should show who pays, why, and how the physical side (crates, pre-cooling, transport) is handled.

## 🚀 Make It Your Own: Hackathon Extensions
Pick one or two. Don't try all of them.
- **Voice in, voice out:** farmers send a WhatsApp voice note (*"my okra has yellow spots"*); Gemini transcribes and answers as audio in Marathi or Telugu.
- **Real documents:** replace the mock library with the real SFA and APEDA web pages or PDFs (convert to markdown with headings, keep the tree index).
- **Second crop:** add a mango or onion specification; the tree index and rule check work the same way.
- **Lot traceability:** generate a lot code + QR label from farmer, village and harvest date (spec A6) and print it on the carton.
- **Demand side:** let the buyer post next month's orders, so farmers can plan plantings (the brief says demand clarity drives investment).
- **Better forecast model:** replace the simple rain/heat rules with historical yields per farmer (scikit-learn regression) once you have pilot data.

## 🎤 Pitching to Judges
- Open with the **roadshow story**: 160 products, positive reception, but quality varied. *"We make the second, third and fiftieth shipment as good as the first."*
- Show the **chart**: fill rate jumping from below target to above it with pre-cooling and pull-forward pickings. One picture for both farmers and buyers.
- Show the **mixed lot** being downgraded with the spec clause cited: *"the AI sees, the rules decide, a human confirms."*
- Spend a third of your time on the **pilot proposal**: KPIs, who pays, how it scales. METI is asking for a pilot, not a demo.

## 🛡️ Responsible AI Checklist
- [ ] Photo grades are **screening only**; a trained human grader confirms, and farmers can appeal a downgrade.
- [ ] Every checklist item cites a section and is checked against a real quote. Still, confirm regulations with SFA and APEDA before shipping.
- [ ] Numbers (fill rate, losses, income) are computed by code from stated assumptions, not by the LLM.
- [ ] Hindi messages must be checked by a native speaker before use, and farmers in Nashik, Guntur and Anand mostly speak Marathi, Telugu and Gujarati.
- [ ] Farmer data (names, phone numbers, plot locations, income) is personal: get consent, store the minimum, and never share it with buyers without permission.
- [ ] Watch for bias: a farmer with a poor phone camera must not get lower grades. Track grade agreement by village and phone type.